In [ ]:
#| default_exp core

# core

> Where a ledger lives, what a record is, and how a line gets onto disk.

A ledger is a folder of JSONL files. `orjson` reads and writes them, one record per line, appended and never rewritten.

Two tiers sit side by side. `ledger/` holds the summary: sessions, steps, file touches, commits. It is small, readable in a diff, and meant to be committed. `detail/` holds what a summary cannot carry: whole tool arguments, whole outputs, the exact lines an edit added. It is gitignored, machine-local, and the ledger stays readable without it.

In [ ]:
#| export
import hashlib, os, time, uuid
from pathlib import Path

import orjson
from fastcore.basics import AttrDict, ifnone
from fastcore.foundation import L

`KINDS` names the five record kinds. Every reader here skips a line whose `kind` is not one of them, so a newer writer can add a kind and an older reader keeps working.

`MAX_LINE` caps a committed line, because a ledger row is a summary and anything longer belongs in `detail`. `MAX_DETAIL` caps a detail line, which is large because a tool result belongs there whole.

In [ ]:
#| export
KINDS = ('session', 'step', 'touch', 'commit', 'note')
LEDGER, DETAIL = 'ledger', 'detail'
DIR = '.panjika'
MAX_LINE = 16_000
MAX_DETAIL = 400_000

## Reading and writing one line

In [ ]:
#| export
def dumps(rec):
    "One record as the bytes of a line, newline included."
    return orjson.dumps(rec, option=orjson.OPT_SORT_KEYS) + b'\n'


def loads(line):
    "One line as a record, or `None` when the line is not a JSON object."
    try: rec = orjson.loads(line)
    except orjson.JSONDecodeError: return None
    return AttrDict(rec) if isinstance(rec, dict) else None

In [ ]:
from fastcore.test import test_eq
r = loads(dumps({'kind': 'note', 'text': 'hello'}))
test_eq(r.kind, 'note')
test_eq(loads(b'not json'), None)
test_eq(loads(b'[1,2]'), None)

In [ ]:
#| export
def now(): return round(time.time(), 3)


def new_id(prefix=''):
    "A short unique id. Records carry one so a union merge can be deduplicated."
    return f'{prefix}{uuid.uuid4().hex[:12]}'


def hashed(data, n=16):
    "A short content hash, for file contents and for the text of a changed line."
    if isinstance(data, str): data = data.encode('utf-8', 'replace')
    return hashlib.blake2b(data, digest_size=n // 2).hexdigest()


def file_hash(path):
    "The hash of a file's bytes, or `''` when it is not there."
    try: return hashed(Path(path).read_bytes())
    except OSError: return ''

In [ ]:
test_eq(hashed('abc'), hashed('abc'))
assert hashed('abc') != hashed('abd')
test_eq(len(hashed('abc')), 16)
test_eq(file_hash('/does/not/exist'), '')

## Where a ledger lives

`find_home` answers the only question a harness hook has to get right on its own: which ledger am I writing to. It walks up from `start` for a folder that already holds one, then for the repository root, and falls back to the home directory for work outside any repository. Every harness in one repository therefore lands on one file without being told where it is.

In [ ]:
#| export
def git_root(start='.'):
    "The repository `start` is inside, or `None`."
    try:
        from gheasy.repo import repo_root
        root = repo_root(str(start))
        return None if root is None else Path(root)
    except Exception: return None


def find_home(start='.'):
    "The ledger folder for `start`: an existing one above it, else its repository, else `~`."
    here = Path(start).resolve()
    for d in (here, *here.parents):
        if (d/DIR).is_dir(): return d/DIR
    root = git_root(here)
    return (root or Path.home())/DIR

## The folder

`Home` is the folder and the shard names inside it. Records are sharded by the month they were written in, which keeps any one file small enough to read whole and small enough to diff.

`init` writes the two dotfiles that make a committed ledger work. `.gitattributes` marks the ledger tier `merge=union`, so two branches that both appended merge to the union of their lines instead of a conflict. Every reader here deduplicates by `id`, which is what makes that safe. `.gitignore` keeps the detail tier out of the repository.

In [ ]:
#| export
GITATTRIBUTES = """# Append-only. Take both sides of a merge; readers deduplicate by record id.
ledger/*.jsonl merge=union
"""

GITIGNORE = """# Machine-local: whole tool arguments, whole outputs, exact changed lines.
detail/
"""


class Home:
    "One ledger folder, and the shards inside it."

    def __init__(self, path=None, start='.'):
        self.path = Path(path) if path else find_home(start)
        self.root = self.path.parent

    def __repr__(self): return f'Home({self.path})'
    def __eq__(self, other): return isinstance(other, Home) and self.path == other.path

    def dir(self, tier): return self.path/tier

    def shard(self, tier=LEDGER, at=None):
        "The file a record written at `at` belongs in."
        stamp = time.strftime('%Y-%m', time.localtime(at if at else now()))
        return self.dir(tier)/f'{stamp}.jsonl'

    def shards(self, tier=LEDGER):
        "Every shard of one tier, oldest first."
        d = self.dir(tier)
        return sorted(d.glob('*.jsonl')) if d.is_dir() else []

    @property
    def exists(self): return self.path.is_dir()

    def init(self):
        "Make the folder and the two dotfiles that let the ledger tier be committed. Idempotent."
        for tier in (LEDGER, DETAIL): self.dir(tier).mkdir(parents=True, exist_ok=True)
        (self.path/'.gitattributes').write_text(GITATTRIBUTES)
        (self.path/'.gitignore').write_text(GITIGNORE)
        return self

## Appending

One record is one `write` syscall to a file opened `O_APPEND`, which is what keeps two harnesses writing at the same moment from interleaving inside a line. A record too long for its tier is clipped rather than refused. Losing the tail of a tool result is better than losing the fact that the call happened, and the longest string field gives way first so the short ones still describe what happened.

In [ ]:
#| export
def clip(rec, limit):
    "Shorten a record until its line fits, longest string field first."
    line = dumps(rec)
    if len(line) <= limit: return rec, len(line)
    rec = dict(rec)
    for k in sorted((k for k, v in rec.items() if isinstance(v, str)),
                    key=lambda k: len(rec[k]), reverse=True):
        over = len(dumps(rec)) - limit
        if over <= 0: break
        keep = max(0, len(rec[k]) - over - 32)
        rec[k] = rec[k][:keep] + f'... [clipped {len(rec[k]) - keep} chars]' if keep else ''
    return rec, len(dumps(rec))


def append(path, rec, limit=MAX_LINE):
    "Write one record as one line. Returns the record as it was written."
    rec, _ = clip(rec, limit)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o644)
    try: os.write(fd, dumps(rec))
    finally: os.close(fd)
    return AttrDict(rec)

The detail tier holds whole tool arguments and whole outputs. Losing the `detail/` line from `.gitignore` is how a repository quietly starts committing them.

In [ ]:
import tempfile
d = Path(tempfile.mkdtemp())
h = Home(d/'.panjika').init()
test_eq((h.path/'.gitattributes').read_text().splitlines()[-1], 'ledger/*.jsonl merge=union')
assert 'detail/' in (h.path/'.gitignore').read_text()
append(h.shard(), {'kind': 'note', 'id': 'n1', 'at': now(), 'text': 'first'})
append(h.shard(), {'kind': 'note', 'id': 'n2', 'at': now(), 'text': 'second'})
test_eq(len(h.shard().read_text().strip().splitlines()), 2)

In [ ]:
big, n = clip({'kind': 'step', 'id': 's1', 'tool': 'Edit', 'summary': 'x' * 40_000}, MAX_LINE)
assert n <= MAX_LINE
test_eq(big['tool'], 'Edit')
assert big['summary'].endswith('chars]')

## Reading

`records` streams every line of a tier, drops anything unreadable, and keeps the first record it sees for any `id`. That last part is what makes `merge=union` safe: a union merge can leave the same line twice, and a rebase can leave it in two shards.

A line that is not a record at all does not stop the ones around it, and neither does a kind this reader has never heard of.

In [ ]:
#| export
def read_shard(path):
    "Every readable record in one shard."
    out = L()
    try: raw = Path(path).read_bytes()
    except OSError: return out
    for line in raw.splitlines():
        if not line.strip(): continue
        rec = loads(line)
        if rec is not None and rec.get('kind') in KINDS and rec.get('id'): out.append(rec)
    return out


def records(home, tier=LEDGER):
    "Every record in a tier, oldest first, deduplicated by id."
    seen, out = set(), L()
    for shard in home.shards(tier):
        for rec in read_shard(shard):
            if rec.id in seen: continue
            seen.add(rec.id)
            out.append(rec)
    return out.sorted(key=lambda r: r.get('at') or 0)

In [ ]:
dup = {'kind': 'note', 'id': 'n1', 'at': now(), 'text': 'first'}
append(h.shard(), dup)
test_eq(len(records(h)), 2)
test_eq(sorted(r.id for r in records(h)), ['n1', 'n2'])

with h.shard().open('a') as f: f.write('{ this is not json\n')
append(h.shard(), {'kind': 'note', 'id': 'n3', 'at': now(), 'text': 'third'})
test_eq(len(records(h)), 3)

append(h.shard(), {'kind': 'something-new', 'id': 'n4', 'at': now()})
test_eq(sorted(r.id for r in records(h)), ['n1', 'n2', 'n3'])

## Two branches, merged

The ledger tier is committed, so two people on two branches append to the same file and then merge. `merge=union` takes both sides instead of raising a conflict, and the reader drops what the union left twice.

In [ ]:
import subprocess

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

g = Path(tempfile.mkdtemp())/'proj'; g.mkdir(parents=True)
_git(g, 'init', '-q', '-b', 'main')
_git(g, 'config', 'user.email', 'a@b.c'); _git(g, 'config', 'user.name', 'Sam')
gh = Home(g/DIR).init()
append(gh.shard(), {'kind': 'note', 'id': 'base', 'at': 1, 'text': 'base'})
_git(g, 'add', '-A'); _git(g, 'commit', '-qm', 'base')

_git(g, 'checkout', '-q', '-b', 'side')
append(gh.shard(), {'kind': 'note', 'id': 'side', 'at': 3, 'text': 'from the side branch'})
_git(g, 'commit', '-aqm', 'side')

_git(g, 'checkout', '-q', 'main')
append(gh.shard(), {'kind': 'note', 'id': 'main', 'at': 2, 'text': 'from main'})
_git(g, 'commit', '-aqm', 'main')

_git(g, 'merge', '-q', '--no-edit', 'side')
test_eq([r.id for r in records(gh)], ['base', 'main', 'side'])

## Folding

A session is not one record. It is every `session` record carrying the same `session`, folded in time order, so the row a reader sees is the sum of what each harness said about it. A hook can append the beginning of a session, a dozen tool calls later, and the end, with no rewrite and no lock between them.

Later values win, but a blank later value never erases an earlier one, so a hook that knows half the facts can append what it knows without wiping what another harness recorded. A field named in `first` keeps its earliest value instead. A session's opening ask and its start time both describe the session, and a harness that writes one record per turn repeats both on every turn.

In [ ]:
#| export
def fold(recs, key='session', first=()):
    "Merge records sharing `key` into one row each, later values winning and `first` fields keeping their earliest."
    out, held = {}, {}
    for rec in recs:
        k = rec.get(key)
        if not k: continue
        row = out.setdefault(k, AttrDict(rec))
        for name, v in rec.items():
            if v is None or v == '' or name == 'id': continue
            if name in first:
                if (k, name) in held: continue
                held[(k, name)] = True
            row[name] = v
    return L(out.values())

In [ ]:
rows = fold([AttrDict(kind='session', id='r1', session='s', at=1, status='open', model='opus'),
             AttrDict(kind='session', id='r2', session='s', at=2, status='done', title='a title', model='')])
test_eq(len(rows), 1)
test_eq(rows[0].status, 'done')
test_eq(rows[0].title, 'a title')
test_eq(rows[0].model, 'opus')

held = fold([AttrDict(kind='session', id='r1', session='s', at=1, prompt='the first ask'),
             AttrDict(kind='session', id='r2', session='s', at=2, prompt='a later ask')],
            first=('prompt',))
test_eq(held[0].prompt, 'the first ask')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()